In [5]:
import mlflow
import os 
import pandas as pd
import numpy as np
from autofeat import AutoFeatClassifier
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv('cleaned_churn_df.csv')
df

,Unnamed: 0,id,customer_id,begin_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,1,2,5575-GNVDE,2017-04-01,One year,No,Mailed check,56.95,1889.50,DSL,...,Yes,No,No,No,Male,0,No,No,No,0
1,2,3,3668-QPYBK,2019-10-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,...,No,No,No,No,Male,0,No,No,No,1
2,4,5,9237-HQITU,2019-09-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,...,No,No,No,No,Female,0,No,No,No,1
3,5,6,9305-CDSKC,2019-03-01,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,...,Yes,No,Yes,Yes,Female,0,No,No,Yes,1
4,6,7,1452-KIOVK,2018-04-01,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,...,No,No,Yes,No,Male,0,No,Yes,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,7035,7036,8456-QDAVC,2018-07-01,Month-to-month,Yes,Bank transfer (automatic),78.70,1495.10,Fiber optic,...,No,No,Yes,No,Male,0,No,No,No,0
4828,7038,7039,6840-RESVB,2018-02-01,One year,Yes,Mailed check,84.80,1990.50,DSL,...,Yes,Yes,Yes,Yes,Male,0,Yes,Yes,Yes,0
4829,7039,7040,2234-XADUH,2014-02-01,One year,Yes,Credit card (automatic),103.20,7362.90,Fiber optic,...,Yes,No,Yes,Yes,Female,0,Yes,Yes,Yes,0
4830,7041,7042,8361-LTMKD,2019-07-01,Month-to-month,Yes,Mailed check,74.40,306.60,Fiber optic,...,No,No,No,No,Male,1,Yes,No,Yes,1


In [7]:
# Признаки и таргет
cat_features = [
    'paperless_billing',
    'payment_method',
    'internet_service',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'gender',
    'senior_citizen',
    'partner',
    'dependents',
    'multiple_lines',
]
num_features = ["monthly_charges", "total_charges"]
features = cat_features + num_features
target = "target"  

# Разделение данных
split_column = "begin_date"
test_size = 0.2
df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    shuffle=False,
)


In [8]:
transformations = ("1/", "log", "abs", "sqrt")


afc = AutoFeatClassifier(categorical_cols=cat_features, transformations=transformations, feateng_steps=1, n_jobs=-1)


X_train_features = afc.fit_transform(X_train, y_train)
X_test_features = afc.transform(X_test)

print(f"Исходное количество признаков:  {X_train.shape[1]}")
print(f"Новое количество признаков:     {X_train_features.shape[1]}")

Исходное количество признаков:  16
Новое количество признаков:     33


Залогируем объектр класса AutoFeatClassifier

In [9]:
TABLE_NAME = "users_churn"  

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)


EXPERIMENT_NAME = "churn_troston" 
RUN_NAME = "preprocessing"

In [11]:
artifact_path = "afc"

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name="your run name", experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    afc_info = mlflow.sklearn.log_model(afc, artifact_path=artifact_path)

In [12]:
print(experiment_id)
print(run_id)

2
f86c4813f8324c4d9c9ee4770eb0f1d8


## Теперь обучим модель с учетом новых признаков и залогируем ее в MLFlow

In [14]:
from catboost import CatBoostClassifier

In [13]:
X_train_features

,monthly_charges,total_charges,cat_paperless_billing_No,cat_paperless_billing_Yes,cat_payment_method_Bank transfer (automatic),cat_payment_method_Credit card (automatic),cat_payment_method_Electronic check,cat_payment_method_Mailed check,cat_internet_service_DSL,cat_internet_service_Fiber optic,...,cat_gender_Male,cat_senior_citizen_0,cat_senior_citizen_1,cat_partner_No,cat_partner_Yes,cat_dependents_No,cat_dependents_Yes,cat_multiple_lines_No,cat_multiple_lines_Yes,1/total_charges
0,117.8,8684.8,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.000115
1,92.45,6440.25,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.000155
2,104.15,7689.95,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.000130
3,108.6,7690.9,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.000130
4,108.05,7532.15,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.000133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3860,66.7,579.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.001727
3861,70.7,553.4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.001807
3862,98.25,560.6,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.001784
3863,75.45,480.75,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.002080


In [17]:
model = CatBoostClassifier( iterations=5000, early_stopping_rounds=10000, eval_metric='Accuracy').fit(X_train_features, y_train, eval_set=(X_test_features, y_test), verbose=100)
y_pred = model.predict(X_test_features)
prediction = model.predict(X_train_features)
y_prob = model.predict_proba(X_test_features)[:,1]

Learning rate set to 0.021973
0:	learn: 0.7697283	test: 0.6587384	best: 0.6587384 (0)	total: 2.73ms	remaining: 13.7s
100:	learn: 0.8028461	test: 0.6204757	best: 0.6608066 (5)	total: 334ms	remaining: 16.2s
200:	learn: 0.8131953	test: 0.5956567	best: 0.6608066 (5)	total: 666ms	remaining: 15.9s
300:	learn: 0.8227684	test: 0.5946225	best: 0.6608066 (5)	total: 993ms	remaining: 15.5s
400:	learn: 0.8292367	test: 0.5956567	best: 0.6608066 (5)	total: 1.32s	remaining: 15.2s
500:	learn: 0.8406210	test: 0.5946225	best: 0.6608066 (5)	total: 1.92s	remaining: 17.2s
600:	learn: 0.8486417	test: 0.5956567	best: 0.6608066 (5)	total: 2.25s	remaining: 16.5s
700:	learn: 0.8556274	test: 0.5946225	best: 0.6608066 (5)	total: 2.59s	remaining: 15.9s
800:	learn: 0.8628719	test: 0.5935884	best: 0.6608066 (5)	total: 2.91s	remaining: 15.3s
900:	learn: 0.8693402	test: 0.5946225	best: 0.6608066 (5)	total: 3.25s	remaining: 14.8s
1000:	learn: 0.8763260	test: 0.5915202	best: 0.6608066 (5)	total: 3.59s	remaining: 14.3s
11

In [18]:
from sklearn.metrics import accuracy_score, roc_auc_score
from mlflow.models import infer_signature

REGISTRY_MODEL_NAME = 'AFCModel'

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_params(model.get_params())

    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test_features)))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, model.predict_proba(X_test_features)[:, 1]))

    signature = infer_signature(X_train_features, model.predict(X_train_features))

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="catboost_model",
        signature=signature,
        registered_model_name=REGISTRY_MODEL_NAME 
    )

print(f"Run ID: {run_id}")
print(f"Model URI: {model_info.model_uri}")

Successfully registered model 'AFCModel'.
2026/04/07 18:43:38 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: AFCModel, version 1


Run ID: f839a0b6df454f6899272733d55942ff
Model URI: runs:/f839a0b6df454f6899272733d55942ff/catboost_model


Created version '1' of model 'AFCModel'.
